# SweaterCo Data Exploration

This notebook provides a lightweight exploration of the Track 1 factory data.

The purpose is not to perform modelling, but to understand the available
fields and data boundaries before designing the deterministic Retrieval,
Judgement, Discovery, and Feasibility tools.

Dataset reference date: **2026-04-01**.

The factory operates Monday to Saturday and is closed on Sundays.

In [28]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
TODAY = pd.Timestamp("2026-04-01")

orders = pd.read_csv(DATA_DIR / "orders.csv")
production = pd.read_csv(DATA_DIR / "production_log.csv")
workshops = pd.read_csv(DATA_DIR / "workshops.csv")

print("Orders:", orders.shape)
print("Production:", production.shape)
print("Workshops:", workshops.shape)

Orders: (120, 12)
Production: (360, 3)
Workshops: (8, 11)


## Orders

The order data supports exact lookup, multi-condition search, due-date
retrieval, activity-based analysis, and downstream order-risk judgement.

In [23]:
print("Status:")
display(orders["status"].value_counts())

print("Current stage:")
display(orders["current_stage"].value_counts())

print("Category:")
display(orders["category"].value_counts())

orders.head()

Status:


status
COMPLETE       86
IN_PROGRESS    34
Name: count, dtype: int64

Current stage:


current_stage
COMPLETE    86
KNITTING    12
ASSEMBLY    10
WASHING     10
PACKING      2
Name: count, dtype: int64

Category:


category
TOPS           85
ACCESSORIES    35
Name: count, dtype: int64

,order_id,customer,product,category,pieces,order_date,due_date,status,current_stage,last_activity_date,completed_date,days_late
0,ORD-057,UrbanThread,Cardigan,TOPS,600,2026-01-01,2026-02-02,COMPLETE,COMPLETE,2026-01-25,2026-01-25,-8.0
1,ORD-076,TrendCart,Crewneck sweater,TOPS,1200,2026-01-02,2026-01-27,COMPLETE,COMPLETE,2026-02-08,2026-02-08,12.0
2,ORD-110,Cotton Club,Scarf,ACCESSORIES,600,2026-01-02,2026-01-24,COMPLETE,COMPLETE,2026-01-27,2026-01-27,3.0
3,ORD-025,Cotton Club,Scarf,ACCESSORIES,400,2026-01-03,2026-01-26,COMPLETE,COMPLETE,2026-01-23,2026-01-23,-3.0
4,ORD-077,UrbanThread,Hoodie,TOPS,1000,2026-01-03,2026-02-07,COMPLETE,COMPLETE,2026-01-29,2026-01-29,-9.0


In [24]:
date_columns = [
    "order_date",
    "due_date",
    "last_activity_date",
    "completed_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

in_progress = orders[orders["status"] == "IN_PROGRESS"].copy()

in_progress["days_to_due"] = (
    in_progress["due_date"] - TODAY
).dt.days

in_progress["days_since_activity"] = (
    TODAY - in_progress["last_activity_date"]
).dt.days

print("In-progress orders:", len(in_progress))

in_progress[
    [
        "order_id",
        "customer",
        "pieces",
        "current_stage",
        "due_date",
        "days_to_due",
        "last_activity_date",
        "days_since_activity",
    ]
].sort_values("days_to_due").head(15)

In-progress orders: 34


,order_id,customer,pieces,current_stage,due_date,days_to_due,last_activity_date,days_since_activity
79,ORD-107,Northwind Apparel,2000,PACKING,2026-03-16,-16,2026-04-01,0
74,ORD-120,TrendCart,1500,ASSEMBLY,2026-03-17,-15,2026-03-30,2
83,ORD-114,Northwind Apparel,2000,PACKING,2026-03-23,-9,2026-03-30,2
86,ORD-083,Maple & Co,1000,WASHING,2026-03-23,-9,2026-03-30,2
84,ORD-002,Cotton Club,1000,WASHING,2026-03-29,-3,2026-03-23,9
97,ORD-093,Maple & Co,100,WASHING,2026-03-29,-3,2026-04-01,0
88,ORD-020,TrendCart,1200,WASHING,2026-03-31,-1,2026-03-30,2
98,ORD-055,Cotton Club,800,ASSEMBLY,2026-03-31,-1,2026-04-01,0
90,ORD-029,Northwind Apparel,1500,WASHING,2026-04-01,0,2026-04-01,0
99,ORD-108,Harbor Knits,2000,ASSEMBLY,2026-04-03,2,2026-03-31,1


## Production

The production log contains one record per stage per day for 90 days.

Sunday output is recorded as zero because the factory is closed.
These zeros should not automatically be interpreted as poor production
performance.

In [25]:
production["date"] = pd.to_datetime(production["date"])

print(
    "Date range:",
    production["date"].min().date(),
    "to",
    production["date"].max().date(),
)

display(production["stage"].value_counts())

production.groupby("stage")["pieces_completed"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1)

Date range: 2026-01-01 to 2026-03-31


stage
KNITTING    90
ASSEMBLY    90
WASHING     90
PACKING     90
Name: count, dtype: int64

,count,mean,median,min,max
stage,,,,,
ASSEMBLY,90,639.7,735.0,0,918
KNITTING,90,768.4,868.0,0,1033
PACKING,90,562.4,640.0,0,746
WASHING,90,602.9,685.5,0,801


In [26]:
production[
    production["date"].dt.dayofweek == 6
].head(8)

,date,stage,pieces_completed
12,2026-01-04,KNITTING,0
13,2026-01-04,ASSEMBLY,0
14,2026-01-04,WASHING,0
15,2026-01-04,PACKING,0
40,2026-01-11,KNITTING,0
41,2026-01-11,ASSEMBLY,0
42,2026-01-11,WASHING,0
43,2026-01-11,PACKING,0


## Workshops

Workshop data provides objective information for downstream feasibility
analysis, including supported categories, status, capacity, queue,
pickup lead time, quality, cost, and batch-size limits.

In [27]:
display(workshops)

print("Status:")
display(workshops["status"].value_counts())

print("Supported categories:")
display(workshops["makes"].value_counts())

,workshop_id,name,capacity_pieces_per_day,pickup_lead_days,defect_rate,cost_per_piece,makes,status,max_batch_pieces,current_queue_days,notes
0,W1,QuickStitch,300,1,0.12,1.5,TOPS,ACTIVE,NaN,4.0,fastest in the region; quality slips under pre...
1,W2,SteadyHands,170,2,0.03,1.3,TOPS+ACCESSORIES,ACTIVE,NaN,2.5,"never early, never late, never sloppy"
2,W3,BudgetWorks,230,3,0.09,0.8,TOPS,ACTIVE,NaN,5.0,cheapest by far
3,W4,Little Loom,70,1,0.02,1.6,ACCESSORIES,ACTIVE,NaN,0.5,"tiny boutique shop, immaculate work"
4,W5,GiantWeave,400,4,0.06,1.1,TOPS,ACTIVE,NaN,3.0,"huge capacity, slow pickup and delivery"
5,W6,Nimble Needle,130,1,0.05,1.2,TOPS+ACCESSORIES,ACTIVE,NaN,1.0,quick turnaround on small batches
6,W7,OldMill,200,2,0.10,0.9,TOPS+ACCESSORIES,SUSPENDED,NaN,0.0,suspended after last month's quality audit
7,W8,FreshStart,100,2,0.04,1.0,TOPS,ACTIVE,300.0,0.0,new workshop on trial: small batches only


Status:


status
ACTIVE       7
SUSPENDED    1
Name: count, dtype: int64

Supported categories:


makes
TOPS                4
TOPS+ACCESSORIES    3
ACCESSORIES         1
Name: count, dtype: int64

## Tool Design Implications

The exploration supports the following deterministic Retrieval tools:

1. Exact order lookup
2. Multi-condition factory order search
3. Production record retrieval and deterministic aggregation
4. Workshop profile retrieval

All retrieval results should retain source-level evidence for Tracing.

Business concepts such as **normal production**, **stalled orders**,
**order risk**, **anomaly severity**, and **feasibility** are not directly
stored in the source data. They should therefore be defined explicitly
in the corresponding Judgement, Discovery, or Feasibility tools rather
than inferred inside Retrieval.